In [ ]:
# ================================
# 07-hybrid-experiment-v2.ipynb
# Phase 5b: Additional hybrid runs with new LR and stability improvements
# ================================

# ------------------------------
# Install missing libraries (bitsandbytes)
# ------------------------------
!pip install -q bitsandbytes

# Fix environment: remove incompatible torchao
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q trl --no-deps
!pip install -q accelerate

import torch, transformers, datasets, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Quick CUDA sanity check
x = torch.randn(100).cuda()
y = torch.randn(100).cuda()
z = torch.matmul(x, y)
print("✓ CUDA ops work:", z.item())

# ------------------------------
# Imports
# ------------------------------
import os
import time
import math
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # <-- correct import
from peft import LoraConfig, IA3Config, TaskType, get_peft_model
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from torch.optim.lr_scheduler import LinearLR

# ... rest of notebook continues as before ...

# ------------------------------
# Configuration
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 16
SEEDS = [42, 123, 456]
BUDGETS = [500, 2000]               # only these budgets need stabilization
HYBRID_LRS = [0.0007, 0.0003]       # new LRs to test (0.0007 first, 0.0003 as backup)
LANGUAGES = ["hi"]                  # Hindi only for now (add "te" if time)

# Paths (same as previous notebooks)
DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ------------------------------
# Helper functions (reused from prior notebooks)
# ------------------------------
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).cuda()

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_loaders(language, budget):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")

    def tokenize(batch):
        return tokenizer(
            batch["premise"], batch["hypothesis"],
            truncation=True, padding="max_length", max_length=MAX_LENGTH
        )

    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")
    train_ds = train_ds.map(tokenize, batched=True)
    valid_ds = valid_ds.map(tokenize, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])
    train_ds.set_format("torch")
    valid_ds.set_format("torch")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=32)
    return train_loader, valid_loader

# ------------------------------
# Hybrid model builder: sequential IA³ → LoRA with verification
# ------------------------------
def build_hybrid_model():
    base = load_base_model()

    # 1. Apply IA³
    ia3_config = IA3Config(
        task_type=TaskType.SEQ_CLS,
        target_modules=["key", "value", "output.dense"],
        feedforward_modules=["output.dense"],
        modules_to_save=["classifier"]
    )
    model = get_peft_model(base, ia3_config)
    print("After IA³: trainable params =", count_trainable_params(model))

    # 2. Apply LoRA on top of IA³
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["query", "value"],
        modules_to_save=["classifier"]
    )
    model = get_peft_model(model, lora_config)
    print("After LoRA: trainable params =", count_trainable_params(model))

    # Verify that both adapters exist
    if hasattr(model, "peft_config"):
        print("Active adapters:", list(model.peft_config.keys()))
    else:
        print("Warning: model does not have peft_config attribute")

    return model

# ------------------------------
# Training and evaluation with gradient clipping and scheduler
# ------------------------------
def train_and_evaluate_hybrid(method, language, budget, seed, lr, epochs):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    train_loader, valid_loader = build_loaders(language, budget)

    # Build hybrid model
    model = build_hybrid_model()
    trainable_params = count_trainable_params(model)

    # Optimizer
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )

    # Compute steps and warmup
    steps_per_epoch = math.ceil(budget / BATCH_SIZE)
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(0.1 * total_steps)

    # Scheduler: linear warmup then constant (no decay to keep it simple)
    scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)

    model.train()
    start_time = time.perf_counter()

    for epoch in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss
            loss.backward()

            # Gradient clipping to prevent explosions
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    train_time = time.perf_counter() - start_time

    # Peak GPU memory measurement (dummy forward)
    torch.cuda.reset_peak_memory_stats()
    model.train()
    with torch.no_grad():
        dummy_batch = next(iter(train_loader))
        dummy_batch = {k: v.cuda() for k, v in dummy_batch.items()}
        _ = model(**dummy_batch)
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

    # Evaluation
    model.eval()
    predictions, labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")

    # Clean up
    del model
    torch.cuda.empty_cache()

    return {
        "method": method,
        "language": language,
        "budget": budget,
        "seed": seed,
        "lr": lr,
        "epochs": epochs,
        "warmup_steps": warmup_steps,
        "total_steps": total_steps,
        "accuracy": round(accuracy, 6),
        "macro_f1": round(macro_f1, 6),
        "trainable_params": trainable_params,
        "peak_gpu_memory_gb": round(peak_memory_gb, 4),
        "training_time_sec": round(train_time, 2)
    }

# ------------------------------
# Main experiment loop
# ------------------------------
results_file = "/kaggle/working/hybrid_v2_results.csv"
results = []

total_runs = len(LANGUAGES) * len(BUDGETS) * len(HYBRID_LRS) * len(SEEDS)
current_run = 0

for language in LANGUAGES:
    for budget in BUDGETS:
        epochs = 10 if budget <= 500 else 5
        for lr in HYBRID_LRS:
            for seed in SEEDS:
                current_run += 1
                print(f"\n[{current_run}/{total_runs}] Hybrid IA³+LoRA | {language} | budget={budget} | lr={lr} | seed={seed}")

                try:
                    result = train_and_evaluate_hybrid(
                        method="ia3_lora_sequential",
                        language=language,
                        budget=budget,
                        seed=seed,
                        lr=lr,
                        epochs=epochs
                    )
                    results.append(result)
                    print(f"  ✓ Acc: {result['accuracy']:.4f}  F1: {result['macro_f1']:.4f}  Time: {result['training_time_sec']:.1f}s")

                    # Save incrementally
                    pd.DataFrame([result]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)

                except Exception as e:
                    print(f"  ✗ ERROR: {e}")
                    error_row = {
                        "method": "ia3_lora_sequential",
                        "language": language,
                        "budget": budget,
                        "seed": seed,
                        "lr": lr,
                        "error": str(e)
                    }
                    pd.DataFrame([error_row]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)

print(f"\n✅ Hybrid v2 experiment completed. {total_runs} configurations attempted.")

# ------------------------------
# Combine with previous hybrid results and generate summary
# ------------------------------
# Load previous hybrid results (if available)
prev_file = "/kaggle/working/hybrid_experiment_results.csv"
if os.path.exists(prev_file):
    prev_df = pd.read_csv(prev_file)
    print(f"Loaded previous hybrid results: {len(prev_df)} rows")
else:
    prev_df = pd.DataFrame()
    print("No previous hybrid results file found.")

# Load new results
if os.path.exists(results_file):
    new_df = pd.read_csv(results_file)
    print(f"Loaded new hybrid results: {len(new_df)} rows")
else:
    new_df = pd.DataFrame()
    print("No new results file found.")

# Combine
all_hybrid = pd.concat([prev_df, new_df], ignore_index=True) if not prev_df.empty else new_df
if not all_hybrid.empty:
    all_hybrid.to_csv("/kaggle/working/all_hybrid_results.csv", index=False)
    print(f"Combined hybrid results saved: {len(all_hybrid)} rows")

    # Summary by (budget, lr)
    summary_hybrid = all_hybrid.groupby(["budget", "lr"]).agg({
        "macro_f1": ["mean", "std", "count"],
        "accuracy": ["mean", "std"]
    }).round(4)
    print("\n=== Summary of All Hybrid Results ===")
    print(summary_hybrid)

    # Count collapses
    collapsed = all_hybrid[np.isclose(all_hybrid["accuracy"], 1/3, atol=0.001)]
    print(f"\nTotal collapses in hybrid: {len(collapsed)} / {len(all_hybrid)}")
    if not collapsed.empty:
        print(collapsed[["budget", "lr", "seed", "accuracy", "macro_f1"]])

# ------------------------------
# Compare with pure methods from original experiment
# ------------------------------
orig_file = "/kaggle/working/experiment_results.csv"
if os.path.exists(orig_file):
    orig_df = pd.read_csv(orig_file)
    # Filter to Hindi only
    orig_hi = orig_df[orig_df["language"] == "hi"]
    # Get best pure per (method, budget)
    pure_best = orig_hi.groupby(["method", "budget"]).agg({
        "macro_f1": ["mean", "max"]
    }).round(4)
    print("\n=== Pure Method Best (Hindi) ===")
    print(pure_best)

    # Compare hybrid best vs pure best at budgets 500 and 2000
    hybrid_best = all_hybrid[all_hybrid["language"] == "hi"].groupby(["budget"]).agg({
        "macro_f1": ["max", "mean"]
    }).round(4)
    print("\n=== Hybrid Best (Hindi) ===")
    print(hybrid_best)

    # Create comparison table
    comp = pd.DataFrame({
        "Budget": [500, 2000],
        "LoRA_F1": [orig_hi[(orig_hi["method"]=="lora") & (orig_hi["budget"]==500)]["macro_f1"].mean(),
                    orig_hi[(orig_hi["method"]=="lora") & (orig_hi["budget"]==2000)]["macro_f1"].mean()],
        "IA3_F1": [orig_hi[(orig_hi["method"]=="ia3") & (orig_hi["budget"]==500)]["macro_f1"].mean(),
                   orig_hi[(orig_hi["method"]=="ia3") & (orig_hi["budget"]==2000)]["macro_f1"].mean()],
        "Hybrid_F1": [all_hybrid[(all_hybrid["budget"]==500)]["macro_f1"].max(),
                      all_hybrid[(all_hybrid["budget"]==2000)]["macro_f1"].max()]
    })
    print("\n=== Comparison Table ===")
    print(comp.to_string(index=False))
else:
    print("Original experiment results not found.")

print("\nAll done.")